# Automated EDA Report Generator

This notebook generates EDA reports using `ydata-profiling`, `sweetviz`, `AutoViz`, and `dtale`.

Usage:
1. Set `dataset_path` to your dataset file.
2. Run the next code cell.
3. Output is saved to `reports/<dataset_name>/`.

If any library is missing, the notebook will skip that report and print a helpful message.

In [ ]:
import os
from pathlib import Path
import pandas as pd

try:
    from ydata_profiling import ProfileReport
except ImportError:
    ProfileReport = None

try:
    import sweetviz
except ImportError:
    sweetviz = None

try:
    from autoviz.AutoViz_Class import AutoViz_Class
except ImportError:
    AutoViz_Class = None

try:
    import dtale
except ImportError:
    dtale = None

def load_dataset(dataset_path: str) -> pd.DataFrame:
    path = Path(dataset_path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    suffix = path.suffix.lower()
    if suffix in {'.csv', '.txt'}:
        for encoding in ['utf-8', 'latin1', 'cp1252']:
            try:
                return pd.read_csv(path, encoding=encoding, engine='python')
            except Exception:
                continue
        raise ValueError(f"Unable to read CSV file with utf-8/latin1/cp1252: {path}")
    if suffix in {'.xls', '.xlsx'}:
        return pd.read_excel(path)
    if suffix == '.json':
        return pd.read_json(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)

    raise ValueError(
        f"Unsupported file extension: {suffix}. Supported: .csv, .txt, .xls, .xlsx, .json, .parquet"
    )

def safe_make_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def generate_eda_reports(dataset_path: str, output_root: str = 'reports') -> Path:
    path = Path(dataset_path)
    output_dir = safe_make_dir(Path(output_root) / path.stem)
    df = load_dataset(dataset_path)

    print(f"Loaded dataset '{dataset_path}' with shape {df.shape}")
    print(f"Saving reports under: {output_dir}")

    result_files = []

    if ProfileReport is not None:
        try:
            report_path = output_dir / f'{path.stem}_ydata_profiling.html'
            profile = ProfileReport(df, explorative=True)
            profile.to_file(report_path)
            result_files.append(report_path)
            print(f"Saved ydata-profiling report: {report_path}")
        except Exception as error:
            print(f"ydata-profiling generation failed: {error}")
    else:
        print('ydata-profiling is not installed. Install it with: pip install ydata-profiling')

    if sweetviz is not None:
        try:
            report_path = output_dir / f'{path.stem}_sweetviz.html'
            report = sweetviz.analyze(df, target_feat=None, pairwise_analysis='auto')
            report.show_html(report_path, open_browser=False)
            result_files.append(report_path)
            print(f"Saved sweetviz report: {report_path}")
        except Exception as error:
            print(f"sweetviz generation failed: {error}")
    else:
        print('sweetviz is not installed. Install it with: pip install sweetviz')

    if AutoViz_Class is not None:
        try:
            report_dir = output_dir / 'autoviz'
            safe_make_dir(report_dir)
            av = AutoViz_Class()
            av.AutoViz(
                filename=str(dataset_path),
                sep=',',
                depVar=None,
                dfte=df,
                header=0,
                verbose=0,
                lowess=False,
                chart_format='svg',
                save_plot_dir=str(report_dir),
                max_rows_analyzed=500000,
                max_cols_analyzed=50,
            )
            print(f"Saved AutoViz plots to: {report_dir}")
        except Exception as error:
            print(f"AutoViz generation failed: {error}")
    else:
        print('AutoViz is not installed. Install it with: pip install autoviz')

    if dtale is not None:
        try:
            app = dtale.show(df, ignore_duplicate=True, subprocess=False)
            if hasattr(app, '_main_url'):
                print(f"dtale started: {app._main_url}")
            elif hasattr(app, 'main_url'):
                print(f"dtale started: {app.main_url}")
            else:
                print('dtale started. Inspect the browser session for the report.')

            if hasattr(dtale, 'save'):
                save_path = output_dir / f'{path.stem}_dtale.html'
                try:
                    dtale.save(app, save_path)
                    result_files.append(save_path)
                    print(f"Saved dtale report: {save_path}")
                except Exception:
                    print('Unable to save dtale report to HTML, but dtale session is running.')
        except Exception as error:
            print(f"dtale startup failed: {error}")
    else:
        print('dtale is not installed. Install it with: pip install dtale')

    if result_files:
        print('Completed report generation. Generated files:')
        for file in result_files:
            print(' -', file)
    else:
        print('Completed run, but no static reports were generated because required libraries were missing or failed.')

    return output_dir


In [ ]:
# Replace this path with your dataset file path and run the cell.
dataset_path = '/path/to/your/dataset.csv'
output_folder = generate_eda_reports(dataset_path)
print(f'Reports folder: {output_folder}')
